# Analyse des données League of Legends

**Problématique** : Analyser et trouver quel est le meilleur champion et quels sont les rôles et les positions les plus représentés

#### Initialisation des librairies

In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

path = "data/"

#### Initialisation des Datasets

##### Champion Dataset

In [2]:
champs = pd.read_csv(path + "champs.csv")

##### Participants Dataset

In [3]:
participants = pd.read_csv(path + "participants.csv")

##### Matches Dataset

In [4]:
matches = pd.read_csv(path + "matches.csv")

##### Stat1 Dataset

In [5]:
stats1 = pd.read_csv(path + "stats1.csv")

##### Stat2 Dataset

In [6]:
stats2 = pd.read_csv(path + "stats2.csv", na_values=[r"\N"], dtype={"wardsbought": "Int64"})

##### Team Bans Dataframe

In [7]:
teambans = pd.read_csv(path + "teambans.csv")

##### Team Stats Dataframe

In [8]:
teamstats = pd.read_csv(path + "teamstats.csv")

#### Tests et analyse

In [9]:
# python
# quick diagnostics
print(participants.shape)
print(participants.columns.tolist())
print(participants['position'].dtype, participants['role'].dtype)
print(participants['position'].isna().sum(), participants['role'].isna().sum())
print(participants[['position', 'role']].head())

# vectorized fix (fast)
mask_bot = participants['position'] == "BOT"
participants.loc[mask_bot & (participants['role'] == 'DUO_SUPPORT'), 'position'] = "SUPP"
participants.loc[mask_bot & (participants['role'] != 'DUO_SUPPORT'), 'position'] = "ADC"

# verify
participants.loc[:, ["position", "role"]].head()

print(participants['championid'].dtype, champs['id'].dtype)


(1834520, 8)
['id', 'matchid', 'player', 'championid', 'ss1', 'ss2', 'role', 'position']
str str
0 0
  position         role
0   JUNGLE         NONE
1      BOT  DUO_SUPPORT
2      BOT    DUO_CARRY
3      TOP         SOLO
4      MID         SOLO
int64 int64


In [13]:
participants["teamid"] = participants.player.apply(lambda x: 100 if x <= 5 else 200)

keep_cols_stats = ["id", "win", "kills"]
keep_cols_participants = ['id', 'player', 'win', 'kills', 'ss1', 'ss2', 'position']

stats = pd.concat([stats1, stats2])

train_data = pd.merge(participants, teamstats.drop(["firstblood"], axis=1), on=["matchid", "teamid"])
train_data = pd.merge(participants, stats, how='left', on=['id'], suffixes=('', '_y'))

champs_info = champs[['id', 'name']].rename(columns={'name': 'champion_name'})

train_data = pd.merge(train_data, champs_info, left_on='championid', right_on='id', how='left', suffixes=('', '_champ'))

train_data = train_data[keep_cols_participants]
only_winners = train_data[train_data["win"] == 1]

only_winners

,id,player,win,kills,ss1,ss2,position
5,14,6,1.0,3.0,11,4,JUNGLE
6,15,7,1.0,4.0,4,12,TOP
7,16,8,1.0,13.0,14,4,MID
8,17,9,1.0,15.0,7,4,ADC
9,18,10,1.0,4.0,14,4,SUPP
...,...,...,...,...,...,...,...
1834510,1865595,1,1.0,3.0,4,12,MID
1834511,1865596,2,1.0,10.0,7,4,ADC
1834512,1865597,3,1.0,22.0,4,11,JUNGLE
1834513,1865598,4,1.0,7.0,12,4,TOP
